# Crossings: did the tracker swap two players where their boxes crossed?

For every place where two boxes overlap, photos of both players just before and just after: shirt
colour (red = team A) and appearance. Claude turns these into automatic fixes in the Pass Review app and
leaves only the unclear crossings for you to check there.

Output per segment in `MyDrive/Playbook/review/match_video_2/<segment>/`: `crossings.csv.gz` and
`crossings_photos.npz`.

**Runtime:** L4 GPU, about 1-2 minutes per segment. **Runtime -> Run all**; do not edit cells.
If it disconnects, **Run all again**: finished segments are skipped.

In [ ]:
# Cell 1 — GPU, Drive, inputs
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.')
print('GPU:', g.stdout.strip())
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
MATCH = 'match_video_2'
CLIPS = sorted(glob.glob(f'{PBD}/video_downloads/{MATCH}_clips/*.mp4'))
REVIEW = f'{PBD}/review/{MATCH}'
MODEL = f'{PBD}/reid_appearance/appearance_model.pt'
NAMES = [os.path.basename(c)[:-4] for c in CLIPS]
missing = [f'{REVIEW}/{n}/tracks_small.csv.gz' for n in NAMES if not os.path.exists(f'{REVIEW}/{n}/tracks_small.csv.gz')]
if missing or not CLIPS:
    raise SystemExit(f'Missing in Drive: {missing[:3] or "the match clips"}')
print(len(CLIPS), 'segments | appearance model:', 'yes' if os.path.exists(MODEL) else 'no (plain DINOv2)')

In [ ]:
# Cell 2 — Our repo and the appearance model
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
import numpy as np, pandas as pd, torch
from tools.crossings import find_crossings, sample_frames, red_share, swap_score
from tools.reid_appearance import Embedder, crops_from_video
model = Embedder()
if os.path.exists(MODEL):
    model.load_state_dict(torch.load(MODEL, map_location=model.device))
print('ready')

In [ ]:
# Cell 3 — Photos around every crossing, colour + appearance, swap score (skips finished segments)
for clip, n in zip(CLIPS, NAMES):
    dst = f'{REVIEW}/{n}/crossings.csv.gz'
    if os.path.exists(dst):
        continue
    t0 = time.time()
    t = pd.read_csv(f'{REVIEW}/{n}/tracks_small.csv.gz')
    t = t[t.class_id.isin((1, 2, 3)) & (t.raw_tracker_id >= 0)]
    X = find_crossings(t, frame='segment_frame')
    s = sample_frames(t, X, frame='segment_frame')
    crops, s = crops_from_video(clip, s)
    emb = model.embed(crops)
    s['red'] = [red_share(c[:, :, ::-1]) for c in crops]            # crops are RGB
    out = []
    for k, r in X.iterrows():
        part = {}
        for side in ('p', 'q'):
            for when in ('b', 'a'):
                m = ((s.k == k) & (s.side == side) & (s.when == when)).to_numpy()
                part[side + when] = {'emb': emb[m], 'red': s.red[m].tolist()} if m.any() else None
        if part['pb'] is None or part['qb'] is None or (part['pa'] is None and part['qa'] is None):
            out.append((np.nan, 'none', np.nan, np.nan)); continue
        sc, basis = swap_score(part['pb'], part['qb'], part['pa'], part['qa'])
        out.append((sc, basis, float(np.mean(part['pb']['red'])), float(np.mean(part['qb']['red']))))
    X[['score', 'basis', 'red_p', 'red_q']] = out if out else np.empty((0, 4))
    X.to_csv('/content/cr.csv.gz', index=False, compression='gzip'); shutil.copy('/content/cr.csv.gz', dst)
    np.savez_compressed('/content/crp.npz', k=s.k.to_numpy(), piece=s.piece.to_numpy(), side=s.side.to_numpy(),
                        when=s.when.to_numpy(), frame=s.frame.to_numpy(), red=s.red.to_numpy(), emb=emb.astype(np.float16))
    shutil.copy('/content/crp.npz', f'{REVIEW}/{n}/crossings_photos.npz')
    left = sum(not os.path.exists(f'{REVIEW}/{m}/crossings.csv.gz') for m in NAMES)
    print(f'{n}: {len(X)} crossings, {int((X.score > 0).sum())} look swapped, {(time.time() - t0) / 60:.1f} min | {left} segments left')
print(f'\nDone. Tell Claude the crossings are in {REVIEW}.')